# 실습 14: 잣대가 둘이면 겹치는 데가 보인다
- 상황: 한 방법으로만 짚은 결과를 그대로 믿기는 어렵다
- 목표: 다르게 생긴 방법으로 한 번 더 짚고, 둘이 같이 가리킨 줄을 찾는다

## Step 0. 앞 실습까지 재현하기

In [10]:
import pandas as pd
from sklearn.ensemble import IsolationForest

# 1. 불러오기
df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

# 2. 센서 열 빈칸을 중앙값으로 채우기
sensor_cols = [c for c in df.columns if c != "result"]
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

# 3. 센서 열만 X, 정답 만들기
X = df[sensor_cols]
정답 = (df["result"] == "불량").astype(int)

# 4. IsolationForest로 지목
탐지기 = IsolationForest(contamination=0.05, random_state=42)
고립지목 = 탐지기.fit_predict(X)

print("이상(-1) 건수:", (고립지목 == -1).sum())
print("그중 불량:", ((고립지목 == -1) & (정답 == 1)).sum())

이상(-1) 건수: 79
그중 불량: 12


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 두 번째 잣대

| 수업에서 쓰는 말 | 정식 이름 | 뜻 |
|---|---|---|
| 이웃과 비교하기 | 국소 이상치 인자 (Local Outlier Factor, LOF) | 내 주변이 얼마나 붐비는지를 보는 방법. 혼자 떨어져 있으면 이상 |
| 이웃 수 | n_neighbors | 몇 명을 이웃으로 볼지. 이것도 사람이 정하는 값 |
| 끝값 | 이상치 (outlier) | 다른 값들보다 유난히 크거나 작은 값. 고립시키기가 잘 잡는 쪽 |
| 무리에서 떨어진 것 | 국소 이상치 (local outlier) | 값 자체는 평범한데 어느 무리에도 안 끼는 줄. 이웃 비교가 잘 잡는 쪽 |
| 겹침 | 교집합 (intersection) | 두 방법이 같이 지목한 줄 |
| 여러 방법을 합쳐 쓰기 | 앙상블 (ensemble) | 한 방법만 믿지 않고 여러 결과를 모아 판단하는 것 |

가운데 칸이 진짜 이름이다. 지난주에 배운 정밀도·재현율처럼 이쪽으로 말해야 통한다.

## Step 2. 이웃과 비교해서 지목하기

In [11]:
# 주변이 얼마나 붐비는지로 이상을 찾는 도구를 불러온다
from sklearn.neighbors import LocalOutlierFactor

# ① 바로 위에서 불러온 도구 이름을 그대로 쓴다
# ② 이웃을 몇 명까지 볼지 정하는 자리
# ③ 몇 %를 이상으로 볼지 정하는 자리 - 앞 실습과 같게 둬야 견줄 수 있다
이웃탐지기 = LocalOutlierFactor(n_neighbors=20, contamination=0.05)

# ④ 여기도 정답은 넣지 않는다
이웃지목 = 이웃탐지기.fit_predict(X)

# ⑤ 이 도구도 앞 실습과 같은 약속을 쓴다
이웃이상 = (이웃지목 == -1)

# ⑥ 두 표시가 둘 다 참인 자리만 남기는 기호
# ⑦ 불량을 무엇으로 적어뒀는지
print("이웃 비교 지목:", 이웃이상.sum(), "건")
print("그중 불량:", (이웃이상 & (정답 == 1 )).sum(), "건")

이웃 비교 지목: 79 건
그중 불량: 5 건


Step 3: 방금 친 코드 정리하기

### 문법 노트 - 두 번째 방법

| 밑줄 | 넣은 것 | 하는 일 | 새로 배운 것인가 |
|---|---|---|---|
| ① | `LocalOutlierFactor` | 주변이 얼마나 붐비는지로 이상을 찾는 도구 | 새로 배움 |
| ② | `n_neighbors` | 이웃을 몇 명까지 볼지 정하는 자리 | 새로 배움 |
| ③ | `contamination` | 전체의 몇 %를 이상으로 볼지 | 앞 실습에서 배웠다 |
| ④ | `X` | 센서 값만 담긴 것 | 계속 쓴 이름 |
| ⑤ | `-1` | 이상이라는 표시 (정상은 `1`) | 앞 실습에서 배웠다 |
| ⑥ | `&` | 양쪽이 다 참인 자리만 참 | 앞 실습에서 배웠다 |
| ⑦ | `1` | 불량을 1로 적어둔 그 값 | 계속 쓰는 약속 |

## Step 4. 두 방법을 나란히

In [12]:
# 두 방법(고립지목, 이웃지목)을 같은 방식으로 채점해 나란히 놓는다
print(f"{'방법':10} {'지목 건수':>8} {'그중 불량':>8} {'지목 중 진짜':>10} {'전체 불량 중 잡은 것':>14}")
for 이름, 지목 in [("고립시키기", 고립지목), ("이웃 비교", 이웃지목)]:
    이상표시 = (지목 == -1)
    잡은것 = (이상표시 & (정답 == 1)).sum()
    지목중진짜 = 잡은것 / 이상표시.sum()
    전체중잡은것 = 잡은것 / (정답 == 1).sum()
    print(f"{이름:10} {이상표시.sum():>8} {잡은것:>8} {round(지목중진짜*100,1):>9}% {round(전체중잡은것*100,1):>13}%")

print()
print("전체 불량률:", round((정답 == 1).mean() * 100, 2), "%")

방법            지목 건수    그중 불량    지목 중 진짜   전체 불량 중 잡은 것
고립시키기            79       12      15.2%          11.5%
이웃 비교            79        5       6.3%           4.8%

전체 불량률: 6.64 %


## Step 5. 둘이 같이 짚은 줄

In [13]:
# ⑧ 앞 실습 방식대로 고립시키기 쪽 표시도 만들어 둔다
고립이상 = (고립지목 == -1)

# ⑨ 두 방법이 같이 지목한 줄만 남기는 기호
둘다 = 고립이상 & 이웃이상

# ⑩ 둘 중 하나라도 지목한 줄을 남기는 기호
둘중하나 = 고립이상 | 이웃이상

print("둘 다 지목:", 둘다.sum(), "건 / 그중 불량", (둘다 & (정답 == 1)).sum(), "건")
print("둘 중 하나라도:", 둘중하나.sum(), "건 / 그중 불량", (둘중하나 & (정답 == 1)).sum(), "건")

# ⑪ 참인 자리가 몇 개인지 세는 것
# ⑫ 소수점 몇 자리에서 끊을지
print("둘 다 지목한 것의 적중률:",
      round((둘다 & (정답 == 1)).sum() / 둘다.sum() * 100, 1), "%")

둘 다 지목: 9 건 / 그중 불량 2 건
둘 중 하나라도: 149 건 / 그중 불량 15 건
둘 다 지목한 것의 적중률: 22.2 %


### 문법 노트 - 두 목록 겹치기

두 표시를 묶는 기호가 둘이다.

    둘 다 참인 자리만    ->  &
    한쪽만 참이어도      ->  |

| 밑줄 | 넣은 것 | 하는 일 | 새로 배운 것인가 |
|---|---|---|---|
| ⑧ | `-1` | 이 도구들이 쓰는 이상 표시 | 앞 실습에서 배웠다 |
| ⑨ | `&` | 양쪽이 다 참인 자리만 참 | 앞 실습에서 배웠다 |
| ⑩ | `\|` | 한쪽이라도 참이면 참 | 새로 배움 |
| ⑪ | `.sum()` | 참인 자리의 개수를 센다 | 계속 쓴 것 |
| ⑫ | `1` | 소수점 첫째 자리까지 남긴다 | 첫날부터 쓴 `round` |

## Step 6. 오늘 알게 된 것

| 방법 | 지목 건수 | 그중 불량 | 지목 중 진짜 |
|---|---|---|---|
| 고립시키기 | [79] | [12] | [15.2%] |
| 이웃 비교 | [79] | [5] | [6.3%] |
| 둘 다 지목 | [9] | [2] | [22.2%] |

- 겹친 목록이 더 진한가 : [22.2%로 고립시키기(15.2%)보다도 높다. 혼자서는 6.3%였던 이웃 비교가 겹치니 값을 했다]
- 겹치면 무엇을 잃나 : [열어볼 건수가 79건에서 [9]건으로 줄었다. 잡은 불량도 2건뿐이라 대부분 놓친다]
- 이 목록을 어디에 쓰나 : [먼저 열어볼 순서. 여기서도 판정이 아니라 후보다]

---
## 직접 해보기 (도전) - 지난주 모델도 같은 줄을 짚었나

- 상황: 답을 보고 배운 지난주 모델과, 답 없이 짚은 오늘 목록이 같은 줄을 가리켰을까
- 할 일: 지난주 모델의 불량 예측과 오늘 겹침 목록을 다시 겹쳐본다
- 결과물: 세 줄짜리 표 1개 + 한 줄 메모

### 1단. 지난주 모델과 겹쳐 보기

In [14]:
# 지난주 모델 - 표준화 + LogisticRegression(가중치), 전체 데이터로 학습하고 전체를 예측한다
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_스케일 = scaler.fit_transform(X)

가중치모델 = LogisticRegression(max_iter=1000, class_weight="balanced")
가중치모델.fit(X_스케일, 정답)
로지스틱예측 = 가중치모델.predict(X_스케일)
로지스틱불량 = (로지스틱예측 == 1)

def 채점(표시, 이름):
    지목건수 = 표시.sum()
    그중불량 = (표시 & (정답 == 1)).sum()
    비율 = 그중불량 / 지목건수 if 지목건수 > 0 else 0
    print(f"{이름:20} 지목 {지목건수:>4}건 / 그중 불량 {그중불량:>3}건 / 지목 중 진짜 {round(비율*100,1)}%")

채점(로지스틱불량, "지난주 모델(로지스틱)")
채점(둘다, "오늘 겹침 목록(둘 다)")
채점(로지스틱불량 & 둘다, "로지스틱 ∩ 겹침 목록")

지난주 모델(로지스틱)         지목  417건 / 그중 불량  78건 / 지목 중 진짜 18.7%
오늘 겹침 목록(둘 다)        지목    9건 / 그중 불량   2건 / 지목 중 진짜 22.2%
로지스틱 ∩ 겹침 목록         지목    6건 / 그중 불량   2건 / 지목 중 진짜 33.3%


### 답을 본 쪽과 안 본 쪽

| 무엇 | 지목 건수 | 그중 불량 | 지목 중 진짜 |
|---|---|---|---|
| 지난주 모델 (답을 보고 배움) | [417] | [78] | [18.7%] |
| 오늘 겹침 목록 (답 안 봄) | [9] | [2] | [22.2%] |
| 셋이 다 가리킨 줄 | [6] | [2] | [33.3%] |

- 알게 된 것 : [근거가 늘수록 진해진다. 22.2% → 33.3%. 대신 9건이 6건으로 줄었다]

### 2단. 이웃 수를 바꿔보기

In [15]:
# 이웃 수만 바꿔가며 이웃 비교를 다시 돌리고, 고립시키기(고립이상)와 겹치는 정도를 본다
print(f"{'이웃 수':8} {'지목 건수':>8} {'그중 불량':>8} {'지목 중 진짜':>10} {'고립시키기와 겹친 건수':>16}")
for k in [5, 20, 50, 100]:
    이웃탐지기 = LocalOutlierFactor(n_neighbors=k, contamination=0.05)
    이웃지목_k = 이웃탐지기.fit_predict(X)
    이웃이상_k = (이웃지목_k == -1)

    지목건수 = 이웃이상_k.sum()
    그중불량 = (이웃이상_k & (정답 == 1)).sum()
    지목중진짜 = 그중불량 / 지목건수
    겹친건수 = (이웃이상_k & 고립이상).sum()

    print(f"{k:<8} {지목건수:>8} {그중불량:>8} {round(지목중진짜*100,1):>9}% {겹친건수:>16}")

이웃 수        지목 건수    그중 불량    지목 중 진짜     고립시키기와 겹친 건수
5              79        5       6.3%               14
20             79        5       6.3%                9
50             79       11      13.9%               42
100            79       11      13.9%               43


### 이웃 수를 바꾸면

(강사 정제본 실측: 5는 14건, 20은 9건, 50은 42건, 100은 43건이 겹쳤다)

| 이웃 수 | 그중 불량 | 지목 중 진짜 | 고립시키기와 겹친 건수 |
|---|---|---|---|
| 5 | [5] | [6.3%] | [14] |
| 20 | [5] | [6.3%] | [9] |
| 50 | [11] | [13.9%] | [42] |
| 100 | [11] | [13.9%] | [43] |

- 알게 된 것 : [이웃을 50명까지 보게 하니 적중이 6.3%에서 13.9%로 두 배가 됐다. 겹친 건수도 9건에서 42건으로 뛰었다]